# 01 · Generate the service-department dataset

**Purpose** — build a dataset that holds up to someone who knows the motor trade,
and that carries its own ground truth.

**Prerequisites** — none. No GPU, no network, no API key.

### Method: reverse generation
The structured record for every update is authored **first**, then the technician
prose is rendered from it. Ground truth therefore exists for 100% of updates at
zero LLM cost — and the free `build.nvidia.com` credit pool stays untouched.

### What makes it realistic
- Work keyed to **Repair Orders**, as shops actually operate
- **Structurally valid VINs** with correct check digits
- The real RO lifecycle, including the blocking states (`AWAITING_AUTHORISATION`, `PARTS_HOLD`)
- Technician notes in the **3 C's** convention (Concern / Cause / Correction)
- Genuine SAE J2012 DTCs, manufacturer-format part numbers, measurements against spec

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))          # notebooks/ -> repo root
os.chdir(os.path.abspath(".."))
from dotenv import load_dotenv; load_dotenv()
print("repo root:", os.getcwd())

## Config

In [ ]:
N_ROS, DAYS, SEED = 400, 14, 20260924
DB = "data/generated/service.sqlite"
print(f"{N_ROS} repair orders over {DAYS} days -> {DB}")

## Execute

In [ ]:
from app.data.generate import build_dataset
stats = build_dataset(db_path=DB, n_ros=N_ROS, days=DAYS, seed=SEED)
print(json.dumps(stats, indent=2))

## What you should see — the catalog

In [ ]:
from app.data.catalog import LABOUR_OPS, CATEGORIES, DTC_CODES, FLEET
import pandas as pd
print(f"{len(LABOUR_OPS)} labour operations across {len(CATEGORIES)} categories")
print(f"{len(DTC_CODES)} diagnostic trouble codes | {len(FLEET)} vehicle models")
pd.DataFrame([{"op_code": o.op_code, "description": o.description,
               "flat_rate_hrs": o.flat_rate_hrs, "category": o.category,
               "safety": o.safety_critical} for o in LABOUR_OPS]).head(12)

## What you should see — a repair order and its updates

In [ ]:
from app.state import db as dbm
con = dbm.connect(DB)
ro = con.execute("SELECT * FROM ros LIMIT 1").fetchone()
print(f"{ro['ro_number']}  {ro['model_year']} {ro['make']} {ro['model']}")
print(f"VIN {ro['vin']}   Reg {ro['registration']}   {ro['odometer_miles']:,} mi")
print(f"Concern: {ro['concern']}")
print(f"{ro['pay_type']} / {ro['wait_type']}   promised {ro['promised_time']}")

In [ ]:
# A technician update, with the ground truth it was rendered from.
u = con.execute("SELECT * FROM updates WHERE ground_truth LIKE '%measurements%' "
                "AND length(text) > 220 LIMIT 1").fetchone()
print("AS WRITTEN BY THE TECHNICIAN:\n")
print(" ", u["text"])
print("\nGROUND TRUTH IT WAS RENDERED FROM:\n")
print(json.dumps(json.loads(u["ground_truth"]), indent=2)[:1200])

## Verify — the realism checks

Every VIN must pass check-digit validation, every op code must resolve to the
catalog, every DTC must be a real code with a valid prefix, and the engine must
fold all 400 repair orders without error.

In [ ]:
!.venv/bin/python scripts/verify_data.py || python scripts/verify_data.py